# Profiling and Automated Validation Notebook — Second Delivery
## Project: Domestic Violence Data Warehouse & Comisarías de Familia API Integration
**Course:** ETL (G01) — Academic Period 2026-2
**Academic Program:** Data Engineering and Artificial Intelligence


In [1]:
import json
import pandas as pd
import numpy as np

# Load raw manifests
with open('../data/raw/primary_source_manifest.json') as f:
    manifest_p = json.load(f)
with open('../data/raw/api_source_manifest.json') as f:
    manifest_a = json.load(f)

print('Primary rows:', manifest_p['row_count'], '| Victims:', manifest_p['total_victims_raw'])
print('API rows:', manifest_a['row_count'], '| Distinct Municipalities:', manifest_a['distinct_municipalities_raw'])


Primary rows: 704551 | Victims: 1501809
API rows: 1249 | Distinct Municipalities: 1102


### 1. Primary Source Profiling Evidence (Policía Nacional - DIJIN)

In [2]:
with open('../docs/evidence/profiling_evidence.json') as f:
    profiling = json.load(f)

print('Measure CANTIDAD profiling:')
print(json.dumps(profiling['primary_source']['measure_profiling'], indent=2))
print('Geographic profiling:')
print(json.dumps(profiling['primary_source']['geographic_profiling'], indent=2))


Measure CANTIDAD profiling:
{
  "attribute": "CANTIDAD",
  "total_victims": 1501809,
  "min": 1.0,
  "max": 198.0,
  "mean": 2.13,
  "invalid_measure_count": 0
}
Geographic profiling:
{
  "distinct_8_digit_dane": 2397,
  "distinct_5_digit_municipalities": 1112,
  "dane_length_distribution": {
    "8": 704551
  },
  "malformed_dane_rows": 6
}


### 2. API Source Profiling Evidence (Comisarías de Familia - MinJusticia)

In [3]:
print('API Municipal Code Profiling:')
print(json.dumps(profiling['api_source']['dane_key_profiling'], indent=2))
print('API Category Distribution:')
print(json.dumps(profiling['api_source']['categoria_municipio_distribution'], indent=2))


API Municipal Code Profiling:
{
  "distinct_raw_codes": 1102,
  "distinct_padded_5_digit_codes": 1102,
  "raw_length_distribution": {
    "5": 1055,
    "4": 194
  },
  "rows_requiring_zero_padding": 194
}
API Category Distribution:
{
  "6": 952,
  "ESP": 99,
  "1": 65,
  "5": 48,
  "2": 45,
  "4": 23,
  "nan": 17
}


### 3. Cross-Source Integration Matching Analysis

In [4]:
print('Integration Matching Metrics:')
print(json.dumps(profiling['cross_source_integration'], indent=2))


Integration Matching Metrics:
{
  "integration_key": "codigo_dane_municipio (5-digit DANE code)",
  "primary_distinct_municipalities": 1112,
  "api_distinct_municipalities": 1102,
  "matched_municipalities_count": 1100,
  "primary_coverage_percentage": 98.92,
  "unmatched_primary_count": 12,
  "unmatched_primary_samples": [
    "00000",
    "52000",
    "88001",
    "91263",
    "91405",
    "91407",
    "91460",
    "91536",
    "91669",
    "91798",
    "94883",
    "97511"
  ],
  "unmatched_api_count": 2,
  "unmatched_api_samples": [
    "27150",
    "94343"
  ],
  "integration_risk_assessment": "Primary codes not in API correspond to non-municipalized areas (corregimientos departamentales in Amazonas, Vaup\u00e9s, Guain\u00eda) and 6 malformed source rows (00000, 52000). Handled via left join with explicit default institution metrics without dropping any primary victim."
}


### 4. Great Expectations Validation Gates Results

In [5]:
with open('../docs/evidence/test_a_successful_run.json') as f:
    test_a = json.load(f)

print('Test A Status:', test_a['overall_status'])
for task in test_a['pipeline_task_executions']:
    print(f"{task['task_id']:35} | {task['state']:10} | {task.get('duration_sec', 0)}s")


Test A Status: PASSED
extract_primary                     | SUCCESS    | 0.458s
validate_primary_raw                | SUCCESS    | 0.416s
extract_api                         | SUCCESS    | 0.005s
validate_api_raw                    | SUCCESS    | 0.032s
transform_and_clean_and_integrate   | SUCCESS    | 1.043s
validate_prepared                   | SUCCESS    | 0.09s
load_dimensional_dw                 | SUCCESS    | 17.171s


### 5. Data Warehouse Reconciliation Verification

In [6]:
with open('../docs/evidence/reconciliation_report.json') as f:
    recon = json.load(f)

print(json.dumps(recon, indent=2))


{
  "load_timestamp": "2026-10-20T18:32:14.041593",
  "status": "SUCCESS",
  "source_reconciliation": {
    "expected_source_records": 704551,
    "loaded_fact_source_records": 704551,
    "records_delta": 0,
    "expected_source_victims": 1501809,
    "loaded_fact_victims": 1501809,
    "victims_delta": 0,
    "exact_reconciliation_pass": true
  },
  "table_row_counts": {
    "dim_tiempo": 6056,
    "dim_geografia": 1113,
    "dim_victima": 11,
    "dim_medio": 6,
    "dim_comisaria": 1247,
    "fact_violencia_intrafamiliar": 701714
  },
  "referential_integrity": {
    "orphan_foreign_keys_count": 0,
    "integrity_pass": true
  }
}
